In [1]:
# Import the pandas library and give it a short nickname 'pd'
import pandas as pd

# Read the CSV file and store it in a variable named 'df' (short for DataFrame)
df = pd.read_csv('ab_data.csv')

# Show the first 5 rows of our data to make sure it loaded correctly
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [2]:
# Check how many total rows and columns we have
print("Total rows and columns before cleaning:", df.shape)

# Find out how many rows have mismatched groups and pages
mismatches = df.query("(group == 'control' and landing_page == 'new_page') or (group == 'treatment' and landing_page == 'old_page')")

print("Number of messy rows we need to delete:", mismatches.shape[0])

Total rows and columns before cleaning: (294478, 5)
Number of messy rows we need to delete: 3893


In [3]:
# Keep ONLY the rows where the group and landing page match correctly
df_clean = df.query("(group == 'control' and landing_page == 'old_page') or (group == 'treatment' and landing_page == 'new_page')")

# Verify the new size of our dataset
print("Total rows after removing mismatches:", df_clean.shape[0])

# Check if any users are in the clean dataset more than once
duplicates = df_clean['user_id'].duplicated().sum()
print("Number of duplicate users:", duplicates)

Total rows after removing mismatches: 290585
Number of duplicate users: 1


In [4]:
# Drop the 1 duplicate user, keeping only their first recorded visit
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')

# Calculate the conversion rate for each group and multiply by 100 for percentages
conversion_rates = df_clean.groupby('group')['converted'].mean() * 100

print("Raw Conversion Rates (%):")
print(conversion_rates)

Raw Conversion Rates (%):
group
control      12.038630
treatment    11.880807
Name: converted, dtype: float64


In [5]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# 1. Count successes (conversions) for each group
convert_old = df_clean.query("group == 'control'")['converted'].sum()
convert_new = df_clean.query("group == 'treatment'")['converted'].sum()

# 2. Count total number of people in each group
n_old = df_clean.query("group == 'control'").shape[0]
n_new = df_clean.query("group == 'treatment'").shape[0]

# 3. Run the Z-test
# count = number of successes, nobs = number of observations
stat, p_value = proportions_ztest([convert_new, convert_old], [n_new, n_old], alternative='larger')

print(f"Z-statistic: {stat:.4f}")
print(f"P-value: {p_value:.4f}")

Z-statistic: -1.3109
P-value: 0.9051


## Executive Summary: A/B Test Results for New Landing Page

**Recommendation:** **DO NOT LAUNCH** the new landing page. 

### Key Findings:
* **Conversion Performance:** The new page had a conversion rate of **11.88%**, which is slightly lower than the old page's **12.04%**.
* **Statistical Rigor:** We conducted a Z-test to determine if this difference was meaningful. The resulting **p-value was 0.9051**.
* **Conclusion:** Because the p-value is significantly higher than our 0.05 threshold, we fail to reject the null hypothesis. There is no evidence that the new page improves user conversion. 

**Strategic Action:** We should retain the current "Old Page" and investigate why the "New Page" failed to engage users. We recommend a fresh design iteration or user interviews before testing again.

In [6]:
from scipy.stats import chisquare

# 1. Get the actual counts of people in each group
observed_counts = df_clean['group'].value_counts().values
# 2. Calculate what the counts SHOULD be (50/50 split)
total_users = df_clean.shape[0]
expected_counts = [total_users / 2, total_users / 2]

# 3. Run a Chi-Square test to see if the difference is "weird"
chi_stat, srm_p_value = chisquare(observed_counts, f_exp=expected_counts)

print(f"Actual Counts: {observed_counts}")
print(f"SRM P-value: {srm_p_value:.4f}")

if srm_p_value < 0.01:
    print("WARNING: Sample Ratio Mismatch detected! The split is not 50/50.")
else:
    print("SUCCESS: No Sample Ratio Mismatch. The experimental design is valid.")

Actual Counts: [145310 145274]
SRM P-value: 0.9468
SUCCESS: No Sample Ratio Mismatch. The experimental design is valid.
